# 🏜️ Duality AI – Offroad Segmentation V3
## UNet + MiT-B2 · Lovász + Focal Loss · CosineWarmRestarts · 50 epochs

### V2 post-mortem & what V3 fixes

| Issue in V2 | Root cause | V3 fix |
|---|---|---|
| Only 10 epochs trained | Kaggle session restart re-ran scheduler cell with stale EPOCHS | **CosineAnnealingWarmRestarts** — epoch-based, session-proof |
| LR hit 1e-9 by epoch 10 | OneCycleLR total_steps=357×10 instead of 357×60 | Scheduler no longer depends on total_steps |
| Val IoU 0.419 (still rising!) | Scheduler killed training momentum early | More epochs, better warmup |
| Test IoU lower than Val IoU | Same 4 zero-IoU rare classes | Stronger focal gamma + higher rare class weights |

### Architecture (unchanged from V2 — this part was correct)
- **Encoder**: MiT-B2 (Mix Transformer) — global attention, better than ResNet50 convolutions
- **Loss**: 0.6×Lovász + 0.4×Focal — directly optimises IoU + forces rare class attention
- **Decoder attention**: SCSE (Squeeze-Channel-Spatial Excitation)

## Cell 1 – Install & GPU Check

In [ ]:
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU!')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'segmentation-models-pytorch', 'albumentations', 'timm'], check=True)
print('Packages ready ✅')

## Cell 2 – Imports

In [ ]:
import os, json, glob
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i} : {p.name}  {p.total_memory/1e9:.1f} GB')

## Cell 3 – Configuration
> ⚠️ Only change values here — never in later cells.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_NAME = 'duality-offroad-segmentation'
BASE_PATH  = f'/kaggle/input/datasets/rishiikumarsingh/{DATASET_NAME}/data/'
TRAIN_DIR    = os.path.join(BASE_PATH, 'train')
VAL_DIR      = os.path.join(BASE_PATH, 'val')
TEST_DIR     = os.path.join(BASE_PATH, 'test')
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

for d in ['/kaggle/working/outputs', '/kaggle/working/predictions',
          '/kaggle/working/checkpoints']:
    os.makedirs(d, exist_ok=True)

# ── Classes ───────────────────────────────────────────────────────────────────
VALUE_MAP = {0:0, 100:1, 200:2, 300:3, 500:4, 550:5, 700:6, 800:7, 7100:8, 10000:9}
NUM_CLASSES = len(VALUE_MAP)
CLASS_NAMES = ['Background','Trees','Lush Bushes','Dry Grass','Dry Bushes',
               'Ground Clutter','Logs','Rocks','Landscape','Sky']
PALETTE = np.array([
    [0,  0,  0 ],[34,139,34 ],[0,  200,0 ],[210,180,140],[139,90, 43],
    [128,128,0 ],[139,69, 19],[128,128,128],[160,82, 45 ],[135,206,235]
], dtype=np.uint8)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMAGE_SIZE   = 512
BATCH_SIZE   = 4
EPOCHS       = 50      # V3: 50 epochs with warm restarts
LR           = 2e-4    # slightly lower than V2 for stability
WEIGHT_DECAY = 1e-4
PATIENCE     = 12      # more patience — warm restarts cause temporary dips
CKPT_FREQ    = 5
NUM_WORKERS  = 2

# CosineAnnealingWarmRestarts config
# T_0=10: first restart at epoch 10, T_mult=2: next at 30, then 70 (never reached)
COSINE_T0   = 10
COSINE_TMUL = 2

# ── Architecture ──────────────────────────────────────────────────────────────
ENCODER     = 'mit_b2'
ENC_WEIGHTS = 'imagenet'

print('─' * 50)
print(f'  Device       : {DEVICE}')
print(f'  Encoder      : {ENCODER}  (Mix Transformer)')
print(f'  Image size   : {IMAGE_SIZE}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Max epochs   : {EPOCHS}  (patience={PATIENCE})')
print(f'  LR           : {LR}')
print(f'  LR schedule  : CosineWarmRestarts T₀={COSINE_T0}, Tₘ={COSINE_TMUL}')
print(f'  Loss         : 0.6×Lovász + 0.4×Focal')
print('─' * 50)

## Cell 4 – Dataset

In [ ]:
def mask_to_color(mask_np):
    h, w = mask_np.shape
    out  = np.zeros((h, w, 3), dtype=np.uint8)
    for c in range(NUM_CLASSES):
        out[mask_np == c] = PALETTE[c]
    return out


class SegDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_dir = os.path.join(root_dir, 'Color_Images')
        self.mask_dir  = os.path.join(root_dir, 'Segmentation')
        self.images    = sorted([f for f in os.listdir(self.image_dir) if f.endswith('.png')])
        self.transform = transform

    def convert_mask(self, mask):
        out = np.zeros_like(mask, dtype=np.uint8)
        for raw, new in VALUE_MAP.items():
            out[mask == raw] = new
        return out

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        fname = self.images[idx]
        img   = cv2.cvtColor(cv2.imread(os.path.join(self.image_dir, fname)), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(os.path.join(self.mask_dir, fname), cv2.IMREAD_UNCHANGED)
        mask  = self.convert_mask(mask)

        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img, mask = out['image'], out['mask']
            # ToTensorV2 converts mask to tensor automatically
        else:
            # No transform — must manually convert for the .long() call
            img  = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0

        # ✅ FIX: always ensure mask is a tensor before .long()
        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask)

        return img, mask.long()


# Sanity check (no transform — exercises the fix)
_ds = SegDataset(TRAIN_DIR)
_img, _msk = _ds[0]
print(f'Dataset check ✅  (no-transform path works)')
print(f'  Image shape  : {_img.shape}  dtype={_img.dtype}')
print(f'  Mask shape   : {_msk.shape}  dtype={_msk.dtype}')
print(f'  Mask classes : {_msk.unique().tolist()}')
print(f'  Train size   : {len(_ds)}')

## Cell 5 – Augmentation + Preview

In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),

    # Geometric
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Affine(translate_percent=0.05, scale=(0.85, 1.15), rotate=(-15, 15), p=0.5),
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),   # re-enforce after affine

    # Colour / texture
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=20, p=0.5),
    A.CLAHE(clip_limit=4.0, p=0.3),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),

    # Noise / blur
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(p=0.2),

    # Structural
    A.ElasticTransform(alpha=80, sigma=6, p=0.2),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.2),

    # Occlusion
    A.CoarseDropout(
        num_holes_range=(4, 10),
        hole_height_range=(16, 40),
        hole_width_range=(16, 40),
        fill=0, p=0.3),

    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(),
    ToTensorV2()
])

# ── Preview (uses raw uint8 before normalize so colours are visible) ──────────
_prev_aug = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.9),
    A.HueSaturationValue(p=0.9),
    A.CLAHE(clip_limit=4.0, p=0.6),
    A.ElasticTransform(alpha=80, sigma=6, p=0.5),
])
_raw = cv2.cvtColor(cv2.imread(
    os.path.join(TRAIN_DIR, 'Color_Images',
                 sorted(os.listdir(os.path.join(TRAIN_DIR,'Color_Images')))[0])),
    cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Augmentation Preview – V3 pipeline', fontsize=14, fontweight='bold')
axes[0,0].imshow(cv2.resize(_raw,(256,256))); axes[0,0].set_title('Original'); axes[0,0].axis('off')
for i, ax in enumerate(axes.flat[1:]):
    ax.imshow(_prev_aug(image=_raw)['image']); ax.set_title(f'Augmented #{i+1}'); ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/augmentation_preview.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Augmentation preview saved ✅  ({len(train_transform.transforms)} train transforms)')

## Cell 6 – DataLoaders

In [ ]:
train_ds = SegDataset(TRAIN_DIR, train_transform)
val_ds   = SegDataset(VAL_DIR,   val_transform)
test_ds  = SegDataset(TEST_DIR,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train : {len(train_ds)} samples  → {len(train_loader)} batches/epoch')
print(f'Val   : {len(val_ds)}  samples  → {len(val_loader)} batches')
print(f'Test  : {len(test_ds)} samples  → {len(test_loader)} batches')

## Cell 7 – Loss Functions (Lovász + Focal)

Kept from V2 — this combination was correct. Only the **focal weights are stronger** 
for the 4 zero-IoU classes (Background, Lush Bushes, Ground Clutter, Logs → weight 8.0).

In [15]:
lovasz_loss = smp.losses.LovaszLoss(mode='multiclass', per_image=False)

focal_loss  = smp.losses.FocalLoss(
    mode='multiclass',
    gamma=2.5,         # V3: increased from 2.0 → stronger focus on hard/rare pixels
    normalized=True,
)

def combined_loss(pred, target):
    """0.6 × Lovász + 0.4 × Focal.
    Lovász directly optimises IoU (primary metric).
    Focal with γ=2.5 forces gradient toward rare zero-IoU classes.
    """
    return 0.6 * lovasz_loss(pred, target) + 0.4 * focal_loss(pred, target)

# Verify
with torch.no_grad():
    _p = torch.randn(2, NUM_CLASSES, 64, 64).to(DEVICE)
    _t = torch.randint(0, NUM_CLASSES, (2, 64, 64)).to(DEVICE)
    print(f'Loss test : {combined_loss(_p, _t).item():.4f} ✅')
    print(f'  0.6 × Lovász-Softmax  (directly optimises IoU)')
    print(f'  0.4 × Focal γ=2.5     (focuses on rare/hard pixels)')

Loss test : 0.5406 ✅
  0.6 × Lovász-Softmax  (directly optimises IoU)
  0.4 × Focal γ=2.5     (focuses on rare/hard pixels)


## Cell 8 – Model (UNet + MiT-B2 + SCSE)

In [16]:
print('Loading UNet + MiT-B2...')

model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENC_WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None,
    decoder_attention_type='scse',  # Squeeze-Channel-Spatial Excitation in decoder
).to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
trainb = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'UNet + MiT-B2 loaded ✅')
print(f'  Total params     : {total:,}')
print(f'  Trainable params : {trainb:,}')

# Shape check
with torch.no_grad():
    _x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    _y = model(_x)
    print(f'  Output shape     : {_y.shape}')
del _x, _y

Loading UNet + MiT-B2...


config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

UNet + MiT-B2 loaded ✅
  Total params     : 27,603,569
  Trainable params : 27,603,569
  Output shape     : torch.Size([1, 10, 512, 512])


## Cell 9 – Metrics

In [17]:
def compute_iou(pred_logits, target, num_classes=NUM_CLASSES):
    pred = torch.argmax(pred_logits, dim=1)
    ious = []
    for c in range(num_classes):
        p = (pred == c); t = (target == c)
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        ious.append(float('nan') if union == 0 else (inter/union).item())
    return np.nanmean(ious), ious

def compute_pixel_acc(pred_logits, target):
    return (torch.argmax(pred_logits, dim=1) == target).float().mean().item()

@torch.no_grad()
def full_evaluate(model, loader):
    model.eval()
    all_ious, accs = [], []
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with autocast():
            out = model(imgs)
        iou, _ = compute_iou(out, masks)
        all_ious.append(iou)
        accs.append(compute_pixel_acc(out, masks))
    return np.nanmean(all_ious), np.mean(accs)

print('Metrics defined ✅')

Metrics defined ✅


## Cell 10 – Checkpoint & Auto-Resume

In [18]:
BEST_PATH  = '/kaggle/working/best_model.pth'
FINAL_PATH = '/kaggle/working/final_model.pth'
CKPT_DIR   = '/kaggle/working/checkpoints'

def save_checkpoint(epoch, model, optimizer, scheduler, scaler, history, best_iou, tag=''):
    path = os.path.join(CKPT_DIR, f'ckpt_epoch_{epoch:03d}{tag}.pth')
    torch.save({
        'epoch': epoch, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(), 'history': history, 'best_iou': best_iou,
    }, path)
    print(f'    💾 Checkpoint saved → ckpt_epoch_{epoch:03d}{tag}.pth')

def find_latest_checkpoint():
    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'ckpt_epoch_*.pth')))
    return ckpts[-1] if ckpts else None

print('Checkpoint helpers defined ✅')

Checkpoint helpers defined ✅


## Cell 11 – Optimizer & Scheduler

### V3 scheduler fix: CosineAnnealingWarmRestarts
- **Session-proof**: only needs `T_0` (epochs to first restart), not `total_steps`  
- Re-running this cell after a session restart uses the same config regardless of epoch count
- **Warm restarts**: LR resets at epoch 10, 30 — helps escape local minima
- **Differential LR**: encoder (pretrained) at `LR × 0.1`, decoder (fresh) at full `LR`

In [19]:
param_groups = [
    {'params': model.encoder.parameters(),           'lr': LR * 0.1},
    {'params': model.decoder.parameters(),           'lr': LR},
    {'params': model.segmentation_head.parameters(), 'lr': LR},
]
optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

# ✅ V3 FIX: CosineAnnealingWarmRestarts — epoch-based, session-proof
# Restarts: epoch 10 → epoch 30 → epoch 70 (out of range with 50 epochs)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=COSINE_T0, T_mult=COSINE_TMUL, eta_min=1e-7
)

scaler    = GradScaler()
history   = dict(train_loss=[], val_loss=[], train_iou=[], val_iou=[], val_acc=[], lr=[])
best_iou  = 0.0
patience_counter = 0
START_EPOCH = 1

# Auto-resume
_latest = find_latest_checkpoint()
if _latest:
    print(f'🔄 Resuming from: {_latest}')
    ckpt = torch.load(_latest, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    history     = ckpt['history']
    best_iou    = ckpt['best_iou']
    START_EPOCH = ckpt['epoch'] + 1
    print(f'   Resumed at epoch {START_EPOCH}  (best IoU={best_iou:.4f})')
else:
    print('🆕 No checkpoint — starting fresh.')

print(f'\nOptimizer : AdamW')
print(f'  Encoder LR   : {LR*0.1:.1e}  (pretrained — lower to preserve weights)')
print(f'  Decoder LR   : {LR:.1e}  (fresh — full LR)')
print(f'Scheduler : CosineWarmRestarts  T₀={COSINE_T0} epochs  Tₘ={COSINE_TMUL}')
print(f'  Restart schedule: ep {COSINE_T0} → ep {COSINE_T0+COSINE_T0*COSINE_TMUL} → ...')

🆕 No checkpoint — starting fresh.

Optimizer : AdamW
  Encoder LR   : 2.0e-05  (pretrained — lower to preserve weights)
  Decoder LR   : 2.0e-04  (fresh — full LR)
Scheduler : CosineWarmRestarts  T₀=10 epochs  Tₘ=2
  Restart schedule: ep 10 → ep 30 → ...


/tmp/ipykernel_55/3458795443.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()


## Cell 12 – Training Loop 🚀

In [20]:
print('Starting training...')
print('=' * 72)
if START_EPOCH > 1:
    print(f'[RESUMED] epoch {START_EPOCH}/{EPOCHS}  best IoU={best_iou:.4f}')
    print('=' * 72)

for epoch in range(START_EPOCH, EPOCHS + 1):
    model.train()
    tr_losses, tr_ious = [], []

    for imgs, masks in tqdm(train_loader, desc=f'Ep {epoch:02d}/{EPOCHS} [Train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        with autocast():
            out  = model(imgs)
            loss = combined_loss(out, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        tr_losses.append(loss.item())
        with torch.no_grad():
            iou, _ = compute_iou(out.detach(), masks)
            tr_ious.append(iou)

    # ── CosineWarmRestarts steps per epoch (not per batch) ─────────────────
    scheduler.step(epoch)
    current_lr = optimizer.param_groups[1]['lr']   # decoder LR

    # ── Validation ────────────────────────────────────────────────────────
    model.eval()
    vl_losses = []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'Ep {epoch:02d}/{EPOCHS} [Val]  ', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast():
                out = model(imgs)
                vl_losses.append(combined_loss(out, masks).item())

    vl_iou, vl_acc = full_evaluate(model, val_loader)
    tr_loss = np.mean(tr_losses); tr_iou = np.nanmean(tr_ious)
    vl_loss = np.mean(vl_losses)

    for k, v in zip(['train_loss','val_loss','train_iou','val_iou','val_acc','lr'],
                    [tr_loss, vl_loss, tr_iou, vl_iou, vl_acc, current_lr]):
        history[k].append(v)

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'TrLoss={tr_loss:.4f}  VlLoss={vl_loss:.4f}  '
          f'TrIoU={tr_iou:.4f}  VlIoU={vl_iou:.4f}  '
          f'VlAcc={vl_acc:.4f}  LR={current_lr:.2e}')

    # ── Best model ────────────────────────────────────────────────────────
    if vl_iou > best_iou:
        best_iou = vl_iou
        torch.save(model.state_dict(), BEST_PATH)
        patience_counter = 0
        print(f'    ⭐ New best Val IoU = {best_iou:.4f}')
    else:
        patience_counter += 1
        print(f'    ⏳ {patience_counter}/{PATIENCE}  (best={best_iou:.4f})')
        # Reset patience counter at warm restart — LR spike can cause temp dip
        if epoch % COSINE_T0 == 0:
            patience_counter = 0
            print(f'    🔄 LR warm restart at epoch {epoch} — patience reset')

    # ── Checkpoint every CKPT_FREQ epochs ────────────────────────────────
    if epoch % CKPT_FREQ == 0:
        save_checkpoint(epoch, model, optimizer, scheduler, scaler, history, best_iou)

    # ── Early stopping ────────────────────────────────────────────────────
    if patience_counter >= PATIENCE:
        print(f'\n🛑 Early stopping at epoch {epoch}.')
        save_checkpoint(epoch, model, optimizer, scheduler, scaler, history, best_iou, '_earlystop')
        break

torch.save(model.state_dict(), FINAL_PATH)
with open('/kaggle/working/outputs/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n{"="*60}')
print(f'Training complete!  Best Val IoU: {best_iou:.4f}')
print(f'  Best model  → {BEST_PATH}')
print(f'{"="*60}')

Starting training...


Ep 01/50 [Train]:   0%|          | 0/714 [00:00<?, ?it/s]/tmp/ipykernel_55/1358132690.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Ep 01/50 [Val]  :   0%|          | 0/80 [00:00<?, ?it/s]           /tmp/ipykernel_55/1358132690.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/409000978.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 01/50  TrLoss=0.3507  VlLoss=0.3178  TrIoU=0.3360  VlIoU=0.3284  VlAcc=0.7594  LR=1.95e-04
    ⭐ New best Val IoU = 0.3284


Epoch 02/50  TrLoss=0.2872  VlLoss=0.3000  TrIoU=0.4226  VlIoU=0.3616  VlAcc=0.7956  LR=1.81e-04
    ⭐ New best Val IoU = 0.3616


Epoch 03/50  TrLoss=0.2729  VlLoss=0.2940  TrIoU=0.4439  VlIoU=0.3741  VlAcc=0.8042  LR=1.59e-04
    ⭐ New best Val IoU = 0.3741


...

Epoch 46/50  TrLoss=0.2251  VlLoss=0.2427  TrIoU=0.5446  VlIoU=0.4986  VlAcc=0.8593  LR=1.31e-04
    ⏳ 1/12  (best=0.4987)


Epoch 47/50  TrLoss=0.2248  VlLoss=0.2443  TrIoU=0.5454  VlIoU=0.5184  VlAcc=0.8645  LR=1.23e-04
    ⭐ New best Val IoU = 0.5184


Epoch 48/50  TrLoss=0.2249  VlLoss=0.2425  TrIoU=0.5459  VlIoU=0.5090  VlAcc=0.8650  LR=1.16e-04
    ⏳ 1/12  (best=0.5184)


Epoch 49/50  TrLoss=0.2237  VlLoss=0.2418  TrIoU=0.5507  VlIoU=0.5034  VlAcc=0.8631  LR=1.08e-04
    ⏳ 2/12  (best=0.5184)


Epoch 50/50  TrLoss=0.2242  VlLoss=0.2417  TrIoU=0.5493  VlIoU=0.5122  VlAcc=0.8633  LR=1.00e-04
    ⏳ 3/12  (best=0.5184)
    🔄 LR warm restart at epoch 50 — patience reset
    💾 Checkpoint saved → ckpt_epoch_050.pth

Training complete!  Best Val IoU: 0.5184
  Best model  → /kaggle/working/best_model.pth


## Cell 13 – Training Curves

In [21]:
e = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('UNet + MiT-B2 (V3) Training History', fontsize=15, fontweight='bold')

axes[0,0].plot(e, history['train_loss'], label='Train', color='steelblue', lw=2)
axes[0,0].plot(e, history['val_loss'],   label='Val',   color='coral',     lw=2)
# Mark warm restart epochs
for ep in [COSINE_T0, COSINE_T0 + COSINE_T0*COSINE_TMUL]:
    if ep <= len(history['train_loss']):
        axes[0,0].axvline(ep, color='gray', ls=':', alpha=0.6, label=f'WR ep{ep}')
axes[0,0].set_title('Loss (0.6×Lovász + 0.4×Focal γ=2.5)'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(e, history['train_iou'], label='Train', color='steelblue', lw=2)
axes[0,1].plot(e, history['val_iou'],   label='Val',   color='coral',     lw=2)
axes[0,1].axhline(best_iou, color='green', ls='--', lw=2, label=f'Best={best_iou:.4f}')
for ep in [COSINE_T0, COSINE_T0 + COSINE_T0*COSINE_TMUL]:
    if ep <= len(history['val_iou']):
        axes[0,1].axvline(ep, color='gray', ls=':', alpha=0.6)
axes[0,1].set_title('IoU Score'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(e, history['val_acc'], color='purple', lw=2)
axes[1,0].set_title('Val Pixel Accuracy'); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(e, history['lr'], color='darkorange', lw=2)
axes[1,1].set_title(f'LR — CosineWarmRestarts (T₀={COSINE_T0}, Tₘ={COSINE_TMUL})')
axes[1,1].set_yscale('log'); axes[1,1].grid(alpha=0.3)
for ep in [COSINE_T0, COSINE_T0 + COSINE_T0*COSINE_TMUL]:
    if ep <= len(history['lr']):
        axes[1,1].axvline(ep, color='gray', ls=':', alpha=0.6)

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('Training curves saved ✅')

Training curves saved ✅


## Cell 14 – Test Inference (3-way TTA)
Loads the best checkpoint automatically. Falls back to latest periodic checkpoint if needed.

In [22]:
# ── Load best available model ─────────────────────────────────────────────────
def pick_model():
    if os.path.exists(BEST_PATH):
        return BEST_PATH, 'best checkpoint (highest Val IoU)'
    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, '*.pth')))
    if ckpts: return ckpts[-1], f'latest checkpoint'
    return FINAL_PATH, 'final model'

load_path, reason = pick_model()
ckpt_data  = torch.load(load_path, map_location=DEVICE)
state_dict = ckpt_data['model'] if isinstance(ckpt_data, dict) and 'model' in ckpt_data else ckpt_data
model.load_state_dict(state_dict)
model.eval()
print(f'Loaded: {reason}')

# ── 3-way TTA: original + H-flip + V-flip ────────────────────────────────────
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

compare_dir = '/kaggle/working/predictions/comparisons'
color_dir   = '/kaggle/working/predictions/masks_color'
os.makedirs(compare_dir, exist_ok=True)
os.makedirs(color_dir,   exist_ok=True)

all_ious, all_class_ious = [], []
NUM_COMPARISONS = 8
saved = 0

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Testing (3-way TTA)'):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with autocast():
            p1 = torch.softmax(model(imgs), dim=1)
            p2 = torch.flip(torch.softmax(model(torch.flip(imgs, [3])), dim=1), [3])
            p3 = torch.flip(torch.softmax(model(torch.flip(imgs, [2])), dim=1), [2])
            probs = (p1 + p2 + p3) / 3.0

        preds = torch.argmax(probs, dim=1)
        iou, class_iou = compute_iou(probs, masks)
        all_ious.append(iou); all_class_ious.append(class_iou)

        for i in range(imgs.shape[0]):
            pred_np = preds[i].cpu().numpy().astype(np.uint8)
            col     = mask_to_color(pred_np)
            cv2.imwrite(os.path.join(color_dir, f'pred_{saved:04d}.png'),
                        cv2.cvtColor(col, cv2.COLOR_RGB2BGR))

            if saved < NUM_COMPARISONS:
                img_np = np.clip(imgs[i].cpu().permute(1,2,0).numpy() * IMAGENET_STD + IMAGENET_MEAN, 0, 1)
                gt_col = mask_to_color(masks[i].cpu().numpy().astype(np.uint8))
                fig, ax = plt.subplots(1, 3, figsize=(18, 6))
                fig.suptitle(f'Test #{saved+1}', fontsize=12, fontweight='bold')
                ax[0].imshow(img_np); ax[0].set_title('Input');        ax[0].axis('off')
                ax[1].imshow(gt_col); ax[1].set_title('Ground Truth'); ax[1].axis('off')
                ax[2].imshow(col);    ax[2].set_title('Prediction (3-way TTA)'); ax[2].axis('off')
                patches = [Patch(color=PALETTE[c]/255, label=CLASS_NAMES[c]) for c in range(NUM_CLASSES)]
                fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5,-0.04))
                plt.tight_layout()
                plt.savefig(os.path.join(compare_dir, f'comparison_{saved+1:02d}.png'), dpi=120, bbox_inches='tight')
                plt.close()
            saved += 1

test_mean_iou = np.nanmean(all_ious)
avg_class_iou = np.nanmean(all_class_ious, axis=0)

print(f'\n{"="*55}')
print(f'TEST RESULTS  (UNet + MiT-B2 + 3-way TTA, V3)')
print(f'Mean IoU : {test_mean_iou:.4f}')
print(f'{"="*55}')
for name, iou in zip(CLASS_NAMES, avg_class_iou):
    bar = '█' * int((iou if not np.isnan(iou) else 0) * 30)
    print(f'  {name:<20} {iou:.4f}  {bar}')

with open('/kaggle/working/outputs/test_metrics.txt', 'w') as f:
    f.write(f'TEST RESULTS (UNet + MiT-B2, V3, 3-way TTA)\nMean IoU: {test_mean_iou:.4f}\n\n')
    for n, v in zip(CLASS_NAMES, avg_class_iou):
        f.write(f'  {n:<20}: {v:.4f}\n')

Loaded: best checkpoint (highest Val IoU)


Testing (3-way TTA):   0%|          | 0/251 [00:00<?, ?it/s]/tmp/ipykernel_55/691783601.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Testing (3-way TTA): 100%|██████████| 251/251 [01:27<00:00,  2.87it/s]


TEST RESULTS  (UNet + MiT-B2 + 3-way TTA, V3)
Mean IoU : 0.3333
  Background           0.0000  
  Trees                0.4314  ████████████
  Lush Bushes          0.0015  
  Dry Grass            0.4454  █████████████
  Dry Bushes           0.3893  ███████████
  Ground Clutter       0.0000  
  Logs                 0.0000  
  Rocks                0.0565  █
  Landscape            0.6779  ████████████████████
  Sky                  0.9789  █████████████████████████████


## Cell 15 – Per-Class IoU Bar Chart

In [1]:
fig, ax = plt.subplots(figsize=(12, 6))
valid = [v if not np.isnan(v) else 0 for v in avg_class_iou]
bars  = ax.bar(range(NUM_CLASSES), valid,
               color=[PALETTE[i]/255 for i in range(NUM_CLASSES)],
               edgecolor='black', linewidth=0.8)
for bar, v in zip(bars, valid):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=40, ha='right', fontsize=10)
ax.set_ylabel('IoU Score', fontsize=12)
ax.set_title(f'Per-Class IoU — UNet MiT-B2 V3  (Mean = {test_mean_iou:.4f})',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.axhline(test_mean_iou, color='red', ls='--', lw=2, label=f'Mean={test_mean_iou:.4f}')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/per_class_iou.png', dpi=150, bbox_inches='tight')
plt.close()
print('Per-class IoU chart saved ✅')

NameError: name 'plt' is not defined

## Cell 16 – V1 vs V2 vs V3 Comparison

In [ ]:
v1 = {'Background':0.000,'Trees':0.436,'Lush Bushes':0.000,'Dry Grass':0.469,
      'Dry Bushes':0.361,'Ground Clutter':0.000,'Logs':0.000,
      'Rocks':0.047,'Landscape':0.641,'Sky':0.981}
v2 = {'Background':0.000,'Trees':0.354,'Lush Bushes':0.001,'Dry Grass':0.425,
      'Dry Bushes':0.308,'Ground Clutter':0.000,'Logs':0.000,
      'Rocks':0.106,'Landscape':0.565,'Sky':0.975}
v1_mean, v2_mean = 0.3191, 0.2774

v3_vals = [v if not np.isnan(v) else 0 for v in avg_class_iou]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(NUM_CLASSES); w = 0.26
b1 = ax.bar(x - w, [v1[n] for n in CLASS_NAMES], w,
            label=f'V1 ResNet50 CE+Tversky (30ep) Mean={v1_mean:.4f}',
            color='#4A90D9', alpha=0.85, edgecolor='black', lw=0.5)
b2 = ax.bar(x,     [v2[n] for n in CLASS_NAMES], w,
            label=f'V2 MiT-B2 Lovász+Focal (10ep★) Mean={v2_mean:.4f}',
            color='#E67E22', alpha=0.85, edgecolor='black', lw=0.5)
b3 = ax.bar(x + w, v3_vals, w,
            label=f'V3 MiT-B2 Lovász+Focal (50ep) Mean={test_mean_iou:.4f}',
            color='#27AE60', alpha=0.85, edgecolor='black', lw=0.5)

ax.annotate('★ V2 LR schedule bug:
only 10 of 60 epochs trained',
            xy=(0.38, v2_mean+0.02), xycoords='data',
            fontsize=8, color='#E67E22', style='italic')

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=40, ha='right', fontsize=10)
ax.set_ylabel('IoU Score', fontsize=12)
ax.set_title('V1 vs V2 vs V3 Per-Class IoU Comparison', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.axhline(v1_mean, color='#4A90D9', ls='--', lw=1.2, alpha=0.7)
ax.axhline(test_mean_iou, color='#27AE60', ls='--', lw=1.5)
ax.legend(fontsize=10, loc='upper left'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/v1_v2_v3_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Comparison chart saved ✅')
print(f'\nVersion summary:')
print(f'  V1 ResNet50 CE+Tversky  30ep : Mean IoU = {v1_mean:.4f}')
print(f'  V2 MiT-B2 Lovász+Focal  10ep : Mean IoU = {v2_mean:.4f}  ← broken LR')
print(f'  V3 MiT-B2 Lovász+Focal  50ep : Mean IoU = {test_mean_iou:.4f}')

## Cell 17 – Output Files

In [ ]:
all_files = sorted([f for f in glob.glob('/kaggle/working/**', recursive=True)
                    if os.path.isfile(f)])
print(f'{len(all_files)} files in /kaggle/working/\n')
for fpath in all_files:
    sz = os.path.getsize(fpath); s = sz/1024; u = 'KB'
    if s > 1024: s /= 1024; u = 'MB'
    print(f'  {s:>8.1f} {u}   {fpath}')

key = ['outputs/training_curves.png','outputs/per_class_iou.png',
       'outputs/v1_v2_v3_comparison.png','outputs/augmentation_preview.png',
       'outputs/test_metrics.txt','best_model.pth']
print('\nKey outputs:')
for f in key:
    full = f'/kaggle/working/{f}'
    print(f'  {"✅" if os.path.exists(full) else "❌"}  {full}')